# 03 - Natural Language Processing (NLP)

**AI-900 Domain:** Describe features of Natural Language Processing workloads on Azure

## What You'll Learn
- **Sentiment Analysis** - Is text positive, negative, or neutral?
- **Key Phrase Extraction** - What are the main topics?
- **Named Entity Recognition (NER)** - Identify people, places, organizations
- **Language Detection** - What language is the text written in?
- **Translation** - Translate text between languages

> **Just click `Run All` - no coding needed!**

## Setup: Load Credentials

In [ ]:
import os
import requests
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

AI_ENDPOINT = os.environ.get("AZURE_AI_ENDPOINT", "").rstrip("/")
AI_KEY = os.environ.get("AZURE_AI_KEY", "")

if not AI_KEY:
    raise ValueError("AZURE_AI_KEY is not set! Please check your .env file.")

print("Credentials loaded successfully!")

---
## Part 1: Sentiment Analysis

Sentiment Analysis determines whether text is **positive**, **negative**, or **neutral**.

Real-world uses:
- Analyzing customer reviews
- Monitoring social media mentions
- Survey response analysis

In [ ]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# Create the Text Analytics client
text_client = TextAnalyticsClient(
    endpoint=AI_ENDPOINT,
    credential=AzureKeyCredential(AI_KEY)
)

print("Text Analytics client created!")

In [ ]:
# Sample texts with different sentiments
reviews = [
    "I absolutely love this product! It exceeded all my expectations.",
    "The service was terrible. I waited 2 hours and nobody helped me.",
    "The meeting is scheduled for 3pm tomorrow in the conference room.",
    "The food was okay, nothing special but not bad either.",
    "Azure AI services are amazing for building intelligent applications!"
]

# Analyze sentiment
results = text_client.analyze_sentiment(documents=reviews)

print("=" * 60)
print("SENTIMENT ANALYSIS RESULTS")
print("=" * 60)

for i, result in enumerate(results):
    if not result.is_error:
        # Emoji indicator
        emoji = {"positive": "[+]", "negative": "[-]", "neutral": "[o]", "mixed": "[~]"}
        indicator = emoji.get(result.sentiment, "[?]")
        
        print(f"\n{indicator} Sentiment: {result.sentiment.upper()}")
        print(f"    Text: \"{reviews[i][:60]}...\"" if len(reviews[i]) > 60 else f"    Text: \"{reviews[i]}\"")
        scores = result.confidence_scores
        print(f"    Scores: positive={scores.positive:.2f} | neutral={scores.neutral:.2f} | negative={scores.negative:.2f}")
    else:
        print(f"\n[!] Error analyzing text {i}: {result.error.message}")

---
## Part 2: Key Phrase Extraction

Key Phrase Extraction identifies the **main talking points** in text. This is useful for:
- Summarizing documents
- Tagging content automatically
- Understanding what a text is about without reading it all

In [ ]:
documents = [
    "Azure AI Foundry provides a unified platform for building, evaluating, and deploying AI models. It supports both pre-built services and custom model training.",
    "The patient was admitted to the hospital with chest pain. The doctor ordered an ECG and blood tests. Results showed no signs of cardiac arrest.",
    "Microsoft's quarterly revenue increased by 15% driven by strong cloud services growth. Azure revenue grew 29% year over year."
]

results = text_client.extract_key_phrases(documents=documents)

print("=" * 60)
print("KEY PHRASE EXTRACTION")
print("=" * 60)

for i, result in enumerate(results):
    if not result.is_error:
        print(f"\nDocument {i + 1}:")
        print(f"  Text: \"{documents[i][:80]}...\"")
        print(f"  Key Phrases: {', '.join(result.key_phrases)}")
    else:
        print(f"\n[!] Error: {result.error.message}")

---
## Part 3: Named Entity Recognition (NER)

NER identifies and classifies **named entities** in text:
- **Person** names
- **Organization** names
- **Location** / geographic entities
- **DateTime** - dates and times
- **Quantity** - numbers, percentages
- And more!

In [ ]:
ner_texts = [
    "Satya Nadella is the CEO of Microsoft, headquartered in Redmond, Washington. The company was founded on April 4, 1975.",
    "The Eiffel Tower in Paris, France attracts about 7 million visitors per year. It was built in 1889 by Gustave Eiffel."
]

results = text_client.recognize_entities(documents=ner_texts)

print("=" * 60)
print("NAMED ENTITY RECOGNITION")
print("=" * 60)

for i, result in enumerate(results):
    if not result.is_error:
        print(f"\nDocument {i + 1}: \"{ner_texts[i][:60]}...\"")
        print(f"{'Entity':<25} {'Category':<15} {'Confidence':<10}")
        print("-" * 50)
        for entity in result.entities:
            print(f"{entity.text:<25} {entity.category:<15} {entity.confidence_score:.2f}")

---
## Part 4: Language Detection

Language Detection identifies what language a text is written in. Azure can detect over 120 languages!

In [ ]:
multilingual_texts = [
    "Hello, how are you today?",
    "Bonjour, comment allez-vous?",
    "Hola, como estas?",
    "Guten Tag, wie geht es Ihnen?",
    "Konnichiwa, ogenki desu ka?",
    "Namaste, aap kaise hain?"
]

results = text_client.detect_language(documents=multilingual_texts)

print("=" * 60)
print("LANGUAGE DETECTION")
print("=" * 60)
print(f"\n{'Text':<40} {'Language':<15} {'Confidence':<10}")
print("-" * 65)

for i, result in enumerate(results):
    if not result.is_error:
        lang = result.primary_language
        text_display = multilingual_texts[i][:38]
        print(f"{text_display:<40} {lang.name:<15} {lang.confidence_score:.2f}")

---
## Part 5: Translation

Azure Translator can translate text between **130+ languages** in real-time.

Note: The Translator API uses the same multi-service key but a different endpoint.

In [ ]:
import uuid

# Translator uses the global endpoint with the multi-service key
translator_endpoint = "https://api.cognitive.microsofttranslator.com"

# Extract region from the AI endpoint (e.g., "eastus" from "https://eastus.api.cognitive.microsoft.com/")
# For multi-service keys, we pass the key and it auto-resolves
headers = {
    "Ocp-Apim-Subscription-Key": AI_KEY,
    "Ocp-Apim-Subscription-Region": os.environ.get("AZURE_SPEECH_REGION", "eastus"),
    "Content-Type": "application/json",
    "X-ClientTraceId": str(uuid.uuid4())
}

# Text to translate
text_to_translate = "Azure AI services make it easy to add intelligence to any application. You can analyze text, images, and speech with just a few lines of code."

# Target languages
target_languages = ["fr", "es", "de", "ja", "hi"]
language_names = {"fr": "French", "es": "Spanish", "de": "German", "ja": "Japanese", "hi": "Hindi"}

# Call Translator API
translate_url = f"{translator_endpoint}/translate?api-version=3.0"
for lang in target_languages:
    translate_url += f"&to={lang}"

body = [{"text": text_to_translate}]
response = requests.post(translate_url, headers=headers, json=body, timeout=10)

print("=" * 60)
print("TRANSLATION RESULTS")
print("=" * 60)
print(f"\nOriginal (English):")
print(f"  \"{text_to_translate}\"\n")

if response.status_code == 200:
    translations = response.json()[0]["translations"]
    for t in translations:
        lang_name = language_names.get(t["to"], t["to"])
        print(f"{lang_name}:")
        print(f"  \"{t['text']}\"\n")
else:
    print(f"Translation failed with status {response.status_code}")
    print(f"Response: {response.text[:300]}")
    print("\nNote: Translation requires a multi-service key with Translator access.")
    print("If you see a 401 error, make sure your Azure AI Services resource includes Translator.")

---
## Key Concepts for AI-900

| NLP Feature | What It Does | Example |
|------------|-------------|--------|
| **Sentiment Analysis** | Detects positive/negative/neutral tone | Customer review analysis |
| **Key Phrase Extraction** | Identifies main topics | Document summarization |
| **Named Entity Recognition** | Finds people, places, organizations | Information extraction |
| **Language Detection** | Identifies what language text is in | Multilingual support |
| **Translation** | Converts text between languages | Global communication |

### AI-900 Exam Tips
- These are all **pre-built NLP services** - no model training needed
- **Azure AI Language** (formerly Text Analytics) handles sentiment, key phrases, NER, and language detection
- **Azure Translator** handles translation (130+ languages)
- Sentiment scores are **confidence values** between 0 and 1
- NER categories include: Person, Organization, Location, DateTime, Quantity, and more
- **Conversational Language Understanding (CLU)** is a related service for building custom intent recognition (not covered here)

---
**Next:** Open `04_speech_ai.ipynb` to explore Speech-to-Text and Text-to-Speech!